In [9]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## Equation of motion derivation

For an object with mass $m$ and moment of inertia matrix $\mathrm{I}_p$ about a pivot which has velocity $\mathbf{v}_p$, where the displacement from the pivot to the center of mass is $\mathbf{r}_{\text{pivot} \rightarrow \text{com}}$, and the object is rotating with angular velocity $\omega$ about the pivot, the kinetic energy is
$$
T = \frac{1}{2} m \mathbf{v}^2_p + m \mathbf{v}_p \cdot (\mathbf{\omega} \times \mathbf{r}_{\text{pivot} \rightarrow \text{com}}) + \frac{1}{2}\mathbf{\omega} \cdot \mathrm{I}_p \mathbf{\omega} .
$$

For our pendulum, we have $\omega = \dot{\theta} \hat{k}$, so the only important component of moment of inertia is $I = \mathrm{I}_p[3,3]$. We also get $\mathbf{\omega} \times \mathbf{r}_{\text{pivot} \rightarrow \text{com}} = \dot\theta l \, \hat\theta$ where $l$ is the distance from the pivot to the center of mass. We construct the pivot so that its velocity only in the $\hat{j}$ direction, so let's call the pivot's vertical position $d$ and velocity $\dot{d}$.

We therefore have
$$
T = \frac{1}{2} m \dot{d}^2 + ml\dot{d}\dot\theta \sin\theta + \frac{1}{2}I\dot{\theta}^2 .
$$

The potential energy is
$$
V = mgy = mg(d-l\cos\theta) .
$$

Plugging these into the Euler-Lagrange equation, we get
$$
\ddot\theta + \frac{ml}{I} (g-\ddot{d})\sin\theta = 0 .
$$

## Reducing variables

If $\ddot{d} = 0$, $\omega_0^2 = \frac{mgl}{I}$ would be the natural frequency of the pendulum. Let's use that and also define $\ddot{d}_g = \ddot{d}/g$, a unitless quantity representing effective pivot acceleration relative to gravity. This lets us rewrite the equation of motion in terms of only a few variables:
$$
\ddot\theta + \omega^2_0 (1-\ddot{d}_g) \sin\theta = 0 .
$$

For the basic Kaptiza pendulum, we have the pivot simply oscillating with amplitude $A$ and frequency $\nu$: $d = A \cos(\nu t)$. Therefore
$$
\ddot{d}_g = -\frac{A}{g} \nu^2 \cos(\nu t)
$$
is the relative acceleration.
This hints towards defining an amplitude relative to gravity $A_g = A/g$, giving us the equation of motion
$$
\ddot\theta + \omega_0^2 \left(1 + A_g \nu^2 \cos(\nu t)\right)\sin\theta = 0
$$
with only three variables describing the experimental setup: $\omega_0$, $A_g$, and $\nu$. Also note that $\omega_0$ is easily measurable by keeping the pivot still and $A_g$ and $\nu$ are chosen by the experimenter.


In [10]:
def angular_acceleration(omega_0_sqr, rel_pivot_acc, theta):
    return -omega_0_sqr * (1 - rel_pivot_acc) * np.sin(theta)

def rel_pivot_acceleration(rel_amplitude, frequency, t):
    return -rel_amplitude * frequency * frequency * np.cos(frequency * t)

In [11]:
def yoshida_4th_step(q, p, t, dt, get_acceleration, get_velocity, params):
    """General 4th-Order Yoshida symplectic integrator step."""
    w1 = 1.0 / (2.0 - 2.0 ** (1.0 / 3.0))
    w0 = -(2.0 ** (1.0 / 3.0)) / (2.0 - 2.0 ** (1.0 / 3.0))

    d = [w1, w0, w1, 0.0]

    curr_q, curr_p = q, p
    curr_t = t

    for i in range(4):
        dt_sub = d[i] * dt
        p_half = curr_p + 0.5 * dt_sub * get_acceleration(curr_q, curr_t, params)
        curr_q = curr_q + dt_sub * get_velocity(p_half, curr_t, params)
        curr_p = p_half + 0.5 * dt_sub * get_acceleration(curr_q, curr_t + dt_sub, params)
        curr_t += dt_sub

    return curr_q, curr_p

In [12]:
omega_0_sqr = np.arange(0.5, 40, 2)
rel_amplitude = np.arange(0.0001, 0.02, 0.001)
frequency = np.arange(20, 300, 10)

w_grid, a_grid, f_grid = np.meshgrid(omega_0_sqr, rel_amplitude, frequency, indexing='ij')

# 2. Stack the flattened grids column-wise into a 2D array of shape (N, 3)
params_2d = np.column_stack((w_grid.ravel(), a_grid.ravel(), f_grid.ravel())).T
params = (params_2d[0], params_2d[1], params_2d[2])


There are three independent variables which affect the motion of the pendulum: 

In [13]:
dt = np.min([1 / np.float64(50.0) / np.max(frequency), 0.001])
t_max = 20.0
steps = int(t_max / dt)
t_steps = np.arange(0, t_max, dt)

In [14]:
def kapitza_velocity(theta_d, t, params):
    return theta_d

def kapitza_acceleration(theta, t, params):
    omega_0_sqr, rel_amplitude, frequency = params
    return angular_acceleration(
        omega_0_sqr, 
        rel_pivot_acceleration(rel_amplitude, frequency, t),
        theta
    )

In [15]:
theta_init = np.ones(shape=params[0].shape[0]) * (np.pi - 0.1)
theta_d_init = np.zeros_like(theta_init)

In [16]:
th, th_d = theta_init, theta_d_init

theta_vals = np.zeros(shape=(theta_init.shape[0], t_steps.shape[0]))


print('Running simulation...')
crossed = np.zeros_like(theta_init)
for i, t in enumerate(t_steps):
    # counts when the pendulum crosses the upwards vertical position
    crossed += ((th >= np.pi) & (theta_vals[:, i-1] < np.pi)) | ((th <= np.pi) & (theta_vals[:, i-1] > np.pi))
    # counts down when the pendulum crosses the downward vertical position
    crossed -= ((th >= 0    ) & (theta_vals[:, i-1] < 0    )) | ((th <= 0    ) & (theta_vals[:, i-1] > 0    ))

    # if crossed is negative, it's oscillating downward. if positive, it's oscillating upward. if zero, it's spinning

    theta_vals[:, i] = th
    th, th_d = yoshida_4th_step(th, th_d, t, dt, kapitza_acceleration, kapitza_velocity, params)


frequencies = crossed / t_max

frequencies_3d = frequencies.reshape(
    omega_0_sqr.shape[0], 
    rel_amplitude.shape[0], 
    frequency.shape[0]
)

Running simulation...


In [1]:
# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Define the interactive plot function using widgets.interact
@widgets.interact(
    freq_index=widgets.IntSlider(
        value=0, 
        min=0, 
        max=len(frequency) - 1, 
        step=1, 
        description='Freq Index:',
        continuous_update=True
    )
)
def update_plot(freq_index):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    current_slice = frequencies_3d[:, :, freq_index]
    cax = ax.pcolormesh(
        rel_amplitude, 
        omega_0_sqr, 
        current_slice, 
        shading='auto', 
        cmap='viridis',
        vmin=np.min(frequencies),
        vmax=np.max(frequencies)
    )
    
    fig.colorbar(cax, ax=ax, label='Frequency / Crossings')
    ax.set_xlabel('Relative Amplitude ($A_g$)')
    ax.set_ylabel('Natural Frequency Squared ($\\omega_0^2$)')
    ax.set_title(f'Frequency Slice at driver $\\nu$ = {frequency[freq_index]:.1f}')
    
    plt.show()

NameError: name 'plt' is not defined

In [18]:
from skimage import measure
import plotly.graph_objects as go

# 1. Run the marching cubes algorithm to extract the isosurface where value == 0
# Note: marching_cubes expects the volume axes order to match (omega_0, amplitude, frequency)
# We pass spacing to scale the coordinates to your real parameter values: (d_omega, d_amp, d_freq)
d_omega = omega_0_sqr[1] - omega_0_sqr[0]
d_amp = rel_amplitude[1] - rel_amplitude[0]
d_freq = frequency[1] - frequency[0]

# Extract isosurface at level = 0.0
verts, faces, normals, values = measure.marching_cubes(
    frequencies_3d, 
    level=0.0, 
    spacing=(d_omega, d_amp, d_freq)
)

# Shift vertices back to match your actual coordinate ranges starting points
verts[:, 0] += omega_0_sqr[0]  # x-axis: omega_0_sqr
verts[:, 1] += rel_amplitude[0] # y-axis: rel_amplitude
verts[:, 2] += frequency[0]    # z-axis: frequency

# 2. Plot the 3D surface interactively using Plotly
fig_3d = go.Figure(data=[
    go.Mesh3d(
        x=verts[:, 0],
        y=verts[:, 1],
        z=verts[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color='lightblue',
        opacity=0.6,
        flatshading=True
    )
])

fig_3d.update_layout(
    title='Isosurface of Kapitza Pendulum Transitions (Stability frequency = 0)',
    scene=dict(
        xaxis_title='Natural Frequency Squared ($\omega_0^2$)',
        yaxis_title='Relative Amplitude ($A_g$)',
        zaxis_title='Pivot Frequency ($\nu$)'
    ),
    width=800,
    height=700
)

fig_3d.show()